# 60 - Checkmate Framework v2 (Fixed)

**Fix:** Never go to 0% allocation. Use signal for RISK ADJUSTMENT, not market timing.

**v2 Position Sizing:**
| MVRV Zone | Original | v2 Fixed |
|-----------|----------|----------|
| < 1.0 (Deep Value) | 100% | 100% |
| 1.0-1.5 (Value) | 75% | 80% |
| 1.5-2.0 (Neutral) | 50% | 60% |
| 2.0-2.5 (Elevated) | 0% ❌ | 40% ✅ |
| 2.5-3.0 (Hot) | -25% ❌ | 30% ✅ |
| > 3.0 (Euphoria) | -50% ❌ | 20% ✅ |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
DATA_DIR = Path.home() / "Documents" / "bitcoin-lab-btc-data-pipeline" / "data" / "daily"

def load_metric(name):
    path = DATA_DIR / f"{name}.parquet"
    if not path.exists(): return pd.DataFrame(columns=['time', 'value'])
    df = pd.read_parquet(path)
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        if df['time'].dt.tz is not None: df['time'] = df['time'].dt.tz_localize(None)
        df = df.set_index('time').sort_index()
    return df

data = {m: load_metric(m) for m in ['price', 'mvrv', 'nupl', 'sopr']}
print("Data loaded")

In [ ]:
# V2 Position sizing - NEVER go to 0%
def position_v2(mvrv):
    """Fixed position sizing with minimum floor."""
    if pd.isna(mvrv): return 0.5
    if mvrv < 1.0:   return 1.00  # Deep value - full
    elif mvrv < 1.5: return 0.80  # Value
    elif mvrv < 2.0: return 0.60  # Neutral
    elif mvrv < 2.5: return 0.40  # Elevated
    elif mvrv < 3.0: return 0.30  # Hot
    else:            return 0.20  # Euphoria - minimum floor, NEVER 0%

# Build dataframe
df = data['price'][['value']].rename(columns={'value': 'price'})
df['returns'] = df['price'].pct_change()
df = df.join(data['mvrv'][['value']].rename(columns={'value': 'mvrv'}))
df['mvrv'] = df['mvrv'].ffill()

# Calculate positions
df['position'] = df['mvrv'].apply(position_v2).shift(1)
df['strat_returns'] = df['position'] * df['returns']
df['equity'] = 100000 * (1 + df['strat_returns']).cumprod()
df['hodl'] = 100000 * (1 + df['returns']).cumprod()

# Results
years = (df.index[-1] - df.index[0]).days / 365
strat_cagr = ((df['equity'].iloc[-1] / 100000) ** (1/years) - 1) * 100
hodl_cagr = ((df['hodl'].iloc[-1] / 100000) ** (1/years) - 1) * 100

print(f"\nRESULTS (v2 Fixed)")
print(f"="*50)
print(f"HODL CAGR:     {hodl_cagr:.1f}%")
print(f"Strategy CAGR: {strat_cagr:.1f}%")
print(f"Avg Position:  {df['position'].mean():.0%}")

In [ ]:
# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].semilogy(df.index, df['hodl'], 'orange', label=f'HODL ({hodl_cagr:.0f}%)')
axes[0].semilogy(df.index, df['equity'], '#3b82f6', label=f'v2 Fixed ({strat_cagr:.0f}%)')
axes[0].set_ylabel('Equity ($)')
axes[0].legend()
axes[0].set_title('Checkmate v2: Fixed Position Sizing', fontsize=14, fontweight='bold')

axes[1].fill_between(df.index, 0, df['position']*100, color='#3b82f6', alpha=0.5)
axes[1].axhline(y=20, color='#ef4444', linestyle='--', label='Minimum floor (20%)')
axes[1].set_ylabel('Position (%)')
axes[1].set_ylim(0, 110)
axes[1].legend()

plt.tight_layout()
plt.show()